In [ ]:
# Cell 1: Install Dependencies (with strict NumPy 1.x pinning to prevent ABI conflicts)
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install -q rank-bm25 sentence-transformers transformers accelerate faiss-cpu pandas scikit-learn

In [ ]:
# Cell 2: Table Structurization and Relational Indexing
import os
import re
import json
import sqlite3
import requests
import pandas as pd
import numpy as np

# 1. Self-healing dataset loader
def load_dataset():
    filename = 'cleaned_tatqa_rag_data.json'
    if os.path.exists(filename):
        print(f"Loading existing '{filename}'...")
        with open(filename, 'r', encoding='utf-8') as f:
            return json.load(f)

    print("Cleaned dataset not found. Fetching raw TAT-QA dev set directly from GitHub...")
    url = "https://raw.githubusercontent.com/NExTplusplus/TAT-QA/master/dataset_raw/tatqa_dataset_dev.json"
    res = requests.get(url)
    if res.status_code != 200:
        raise RuntimeError(f"Failed to fetch dataset. HTTP Status: {res.status_code}")

    tatqa_raw = res.json()
    cleaned = []
    for report in tatqa_raw:
        table = report.get('table', {}).get('table', [])
        paras = {p['order']: p['text'] for p in report.get('paragraphs', [])}
        for q in report.get('questions', []):
            rel_text = [paras[i] for i in q.get('rel_paragraphs', []) if i in paras]
            rel_rows = []
            if table:
                rel_rows.append(table[0])
                for reg in q.get('rel_table_regions', []):
                    r_idx = reg[0]
                    if r_idx < len(table) and r_idx != 0 and table[r_idx] not in rel_rows:
                        rel_rows.append(table[r_idx])

            scale = str(q.get('scale', '')).lower()
            mult = 1.0
            if scale == 'thousand': mult = 1000.0
            elif scale == 'million': mult = 1_000_000.0
            elif scale == 'billion': mult = 1_000_000_000.0
            elif scale == 'percent': mult = 0.01

            cleaned.append({
                'id': q.get('uid'),
                'metadata': {
                    'scale_string': scale,
                    'scale_multiplier': mult,
                    'answer_type': q.get('answer_type', 'unknown'),
                    'report_context': report.get('table', {}).get('uid', 'unknown_report')
                },
                'question': q.get('question'),
                'evidence': {
                    'text': " | ".join(rel_text),
                    'table': rel_rows
                },
                'ground_truth': {
                    'answer': q.get('answer'),
                    'derivation': q.get('derivation', '')
                }
            })
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(cleaned, f, indent=4)
    return cleaned

dataset = load_dataset()
eval_subset = dataset[:30]

# 2. Convert Tables to DataFrames and In-Memory SQLite Databases
def parse_table_to_df(table_matrix):
    if not table_matrix or len(table_matrix) < 2:
        return pd.DataFrame()
    raw_headers = [str(c).strip() for c in table_matrix[0]]
    # Sanitize duplicate or empty column headers
    clean_headers = []
    seen = {}
    for i, h in enumerate(raw_headers):
        h_name = h if h else f"col_{i}"
        h_name = re.sub(r'\W+', '_', h_name).strip('_')
        count = seen.get(h_name, 0)
        seen[h_name] = count + 1
        clean_headers.append(f"{h_name}_{count}" if count > 0 else h_name)

    rows = table_matrix[1:]
    clean_rows = []
    for r in rows:
        row_data = [str(c).strip() for c in r]
        if len(row_data) < len(clean_headers):
            row_data += [""] * (len(clean_headers) - len(row_data))
        clean_rows.append(row_data[:len(clean_headers)])

    return pd.DataFrame(clean_rows, columns=clean_headers)

corpus_records = {}
db_conn = sqlite3.connect(":memory:")

year_pattern = re.compile(r'\b(19\d\d|20\d\d)\b')

for item in dataset:
    qid = item['id']
    t_df = parse_table_to_df(item['evidence']['table'])

    # Store table in SQLite with table name matching qid
    safe_table_name = f"tbl_{re.sub(r'[^a-zA-Z0-9]', '_', qid)}"
    if not t_df.empty:
        t_df.to_sql(safe_table_name, db_conn, if_exists='replace', index=False)

    # Extract metadata for pre-filtering
    all_content = f"{item['evidence']['text']} " + " ".join(t_df.values.flatten() if not t_df.empty else [])
    years = list(set(year_pattern.findall(all_content)))

    corpus_records[qid] = {
        'id': qid,
        'text': item['evidence']['text'],
        'table_df': t_df,
        'sql_table_name': safe_table_name,
        'serialized': f"Text: {item['evidence']['text']}\nTable:\n{t_df.to_markdown(index=False) if not t_df.empty else 'No table'}",
        'years': years,
        'scale_multiplier': item['metadata']['scale_multiplier']
    }

print(f"Indexed {len(corpus_records)} financial records with relational schema and temporal tags.")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# Cell 3: Hybrid RAG with HyDE and Cross-Encoder Reranker
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Embedding and Reranking Models...")
dense_embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)

print("Loading Qwen-2.5-3B-Instruct for Planner and HyDE...")
llm_model_id = "Qwen/Qwen2.5-3B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

def generate_local_llm(prompt, max_new_tokens=128):
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id
        )
    return llm_tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# Build Retrieval Corpora
doc_ids = list(corpus_records.keys())
serialized_docs = [corpus_records[did]['serialized'] for did in doc_ids]

# BM25 Tokenization
tokenized_bm25 = [doc.lower().split() for doc in serialized_docs]
bm25_index = BM25Okapi(tokenized_bm25)

# FAISS Dense Index
doc_vectors = dense_embedder.encode(serialized_docs, convert_to_tensor=True, show_progress_bar=False)

def generate_hyde_doc(query):
    prompt = (
        f"<|im_start|>system\nYou are an SEC 10-K financial reporting assistant. Write a short, hypothetical 1-2 sentence excerpt "
        f"from a corporate financial statement that directly answers the question.<|im_end|>\n"
        f"<|im_start|>user\nQuestion: {query}<|im_end|>\n<|im_start|>assistant\n"
    )
    return generate_local_llm(prompt, max_new_tokens=64)

def advanced_hybrid_retrieve(query, top_k_fusion=20, final_k=3, use_hyde=True, year_filter=None):
    # 1. HyDE Transformation
    search_query = f"{query} {generate_hyde_doc(query)}" if use_hyde else query

    # 2. Sparse (BM25) Retrieval
    bm25_scores = bm25_index.get_scores(search_query.lower().split())
    bm25_rankings = np.argsort(-bm25_scores)

    # 3. Dense (MiniLM) Retrieval
    q_vec = dense_embedder.encode(search_query, convert_to_tensor=True)
    dense_scores = (doc_vectors @ q_vec).cpu().numpy()
    dense_rankings = np.argsort(-dense_scores)

    # 4. Reciprocal Rank Fusion (RRF, k=60) with Optional Temporal Filtering
    rrf_scores = {}

    for rank, idx in enumerate(bm25_rankings[:50]):
        did = doc_ids[idx]
        if year_filter and not any(y in corpus_records[did]['years'] for y in year_filter):
            continue
        rrf_scores[did] = rrf_scores.get(did, 0.0) + (1.0 / (60 + rank + 1))

    for rank, idx in enumerate(dense_rankings[:50]):
        did = doc_ids[idx]
        if year_filter and not any(y in corpus_records[did]['years'] for y in year_filter):
            continue
        rrf_scores[did] = rrf_scores.get(did, 0.0) + (1.0 / (60 + rank + 1))

    if not rrf_scores:
        # Fallback if hard filter excludes all documents
        return [doc_ids[i] for i in dense_rankings[:final_k]]

    fused_candidates = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k_fusion]

    # 5. Cross-Encoder Reranking
    pairs = [[query, corpus_records[did]['serialized']] for did in fused_candidates]
    cross_scores = reranker.predict(pairs)
    ranked_top_k = [fused_candidates[i] for i in np.argsort(-cross_scores)[:final_k]]

    return ranked_top_k

In [ ]:
# Cell 4: Planner Agent and Deterministic Execution Sandbox
import math

class FinancialExecutionSandbox:
    @staticmethod
    def safe_execute_python(code_str, table_df=None):
        """Executes arithmetic and Pandas manipulations inside an isolated scope."""
        local_scope = {
            "df": table_df.copy() if table_df is not None else pd.DataFrame(),
            "pd": pd,
            "np": np,
            "result": None
        }
        safe_builtins = {
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow, "len": len, "float": float, "int": int
        }
        clean_code = re.sub(r"```(?:python)?|```", "", code_str).strip()

        # Block dangerous functions
        if any(banned in clean_code for banned in ['import', 'os', 'sys', '__', 'eval', 'exec', 'open', 'subprocess']):
            return None, "Security Violation"

        try:
            exec(clean_code, {"__builtins__": safe_builtins}, local_scope)
            res = local_scope.get("result")
            if res is None:
                match = re.search(r"result\s*=\s*(.+)", clean_code)
                if match:
                    res = eval(match.group(1), {"__builtins__": safe_builtins}, local_scope)
            return float(res) if res is not None else None, "Success"
        except Exception as e:
            return None, str(e)

    @staticmethod
    def safe_execute_sql(query_str, db_connection):
        clean_query = re.sub(r"```(?:sql)?|```", "", query_str).strip()
        if any(banned in clean_query.lower() for banned in ['drop', 'insert', 'update', 'delete', 'alter']):
            return None, "Illegal Write Operation"
        try:
            cursor = db_connection.cursor()
            cursor.execute(clean_query)
            row = cursor.fetchone()
            return (float(row[0]) if row and row[0] is not None else None), "Success"
        except Exception as e:
            return None, str(e)

# Agentic Planner: Query Decomposition and Routing
def planner_agent(question):
    prompt = (
        f"<|im_start|>system\nYou are a financial query planner. Analyze the question and output a JSON object with:\n"
        f"1. 'action': 'lookup' (single value from text/table) or 'calculate' (math/ratio/percentage).\n"
        f"2. 'years': list of years mentioned (e.g. ['2019', '2018']).\n"
        f"3. 'sub_questions': list of atomic questions needed to formulate the answer.\n"
        f"Output JSON ONLY.<|im_end|>\n"
        f"<|im_start|>user\nQuestion: {question}<|im_end|>\n<|im_start|>assistant\n"
    )
    raw = generate_local_llm(prompt, max_new_tokens=96)
    try:
        json_str = re.search(r'\{.*\}', raw, re.DOTALL).group(0)
        plan = json.loads(json_str)
    except Exception:
        years = year_pattern.findall(question)
        action = 'calculate' if any(w in question.lower() for w in ['percent', 'rate', 'change', 'total', 'increase', 'decrease', 'average']) else 'lookup'
        plan = {'action': action, 'years': years, 'sub_questions': [question]}
    return plan

# Program-of-Thoughts (PoT) Generator
def generate_pot_solution(question, context, table_df=None):
    df_desc = f"Available Pandas DataFrame `df` with columns: {list(table_df.columns)}\nData:\n{table_df.to_string()}" if table_df is not None and not table_df.empty else "No structured table."
    prompt = (
        f"<|im_start|>system\nYou are a financial code engine. Compute the answer using deterministic Python. "
        f"Write executable Python code that computes the solution and assigns the final numerical scalar to `result`.\n"
        f"Do not write conversational explanations.\n"
        f"{df_desc}<|im_end|>\n"
        f"<|im_start|>user\nEvidence Context:\n{context}\n\nQuestion: {question}<|im_end|>\n<|im_start|>assistant\n```python\n"
    )
    gen = generate_local_llm(prompt, max_new_tokens=128)
    code = gen.split("```")[0].strip()
    return f"```python\n{code}\n```"

In [ ]:
# Cell 5: End-to-End Evaluation Engine
import string
from collections import Counter

def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    return ' '.join(s.split())

def compute_em(pred, truth):
    return int(normalize_text(pred) == normalize_text(truth))

def compute_f1(pred, truth):
    pred_tokens = normalize_text(pred).split()
    truth_tokens = normalize_text(truth).split()
    if not pred_tokens or not truth_tokens:
        return int(pred_tokens == truth_tokens)
    common = Counter(pred_tokens) & Counter(truth_tokens)
    n_common = sum(common.values())
    if n_common == 0: return 0
    p = 1.0 * n_common / len(pred_tokens)
    r = 1.0 * n_common / len(truth_tokens)
    return (2 * p * r) / (p + r)

def extract_first_number(s):
    nums = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", str(s).replace(',', ''))
    return float(nums[0]) if nums else None

def is_close(val1, val2, tol=1e-2):
    if val1 is None or val2 is None:
        return False
    return math.isclose(val1, val2, rel_tol=tol, abs_tol=tol)

retrieval_records = []
execution_records = []

print("Running Comprehensive Evaluation over Target Cohort...")

for item in eval_subset:
    qid = item['id']
    question = item['question']
    gt_answer = item['ground_truth']['answer']
    gt_val = extract_first_number(gt_answer)
    target_id = qid

    # 1. Execute Planner Agent
    plan = planner_agent(question)

    # 2. Execute Hybrid Retrieval with HyDE and Reranker
    retrieved_ids = advanced_hybrid_retrieve(
        query=question,
        top_k_fusion=20,
        final_k=3,
        use_hyde=True,
        year_filter=plan.get('years', None)
    )

    # Track Retrieval Metrics
    hit = 1 if target_id in retrieved_ids else 0
    rank = retrieved_ids.index(target_id) + 1 if hit else 0
    mrr = (1.0 / rank) if hit else 0.0
    ndcg = (1.0 / math.log2(rank + 1)) if hit else 0.0
    wrong_evidence = 1 if (retrieved_ids and retrieved_ids[0] != target_id) else 0

    retrieval_records.append({
        'qid': qid,
        'hit': hit,
        'precision@3': hit / 3.0,
        'mrr': mrr,
        'ndcg@3': ndcg,
        'wrong_evidence': wrong_evidence
    })

    # 3. Assemble Retrieved Context
    top_record = corpus_records[retrieved_ids[0]]
    combined_context = top_record['serialized']
    target_table_df = top_record['table_df']

    # 4. Route Execution via Action Taxonomy
    if plan.get('action') == 'calculate':
        pot_code = generate_pot_solution(question, combined_context, target_table_df)
        computed_res, status = FinancialExecutionSandbox.safe_execute_python(pot_code, target_table_df)
        success = 1 if status == "Success" and computed_res is not None else 0
        final_pred = str(round(computed_res, 4)) if success else "Calculation Failed"
        num_match = 1 if is_close(computed_res, gt_val) else 0
        arithmetic_err = 1 if (success and not num_match) else 0
    else:
        # Direct Span Lookup
        prompt_lookup = (
            f"<|im_start|>system\nExtract ONLY the exact phrase or scalar from the context answering the question.<|im_end|>\n"
            f"<|im_start|>user\nContext:\n{combined_context}\n\nQuestion: {question}<|im_end|>\n<|im_start|>assistant\n"
        )
        final_pred = generate_local_llm(prompt_lookup, max_new_tokens=32)
        pred_val = extract_first_number(final_pred)
        num_match = 1 if is_close(pred_val, gt_val) else 0
        success = 1
        arithmetic_err = 0

    execution_records.append({
        'qid': qid,
        'em': compute_em(final_pred, str(gt_answer)),
        'f1': compute_f1(final_pred, str(gt_answer)),
        'numerical_match': num_match,
        'arithmetic_error': arithmetic_err,
        'tool_success': success
    })

# Summary Tables
df_retrieval_summary = pd.DataFrame(retrieval_records)
df_execution_summary = pd.DataFrame(execution_records)

benchmark_table = pd.DataFrame([
    {
        'Module': 'Advanced Retrieval (HyDE + RRF + Cross-Encoder)',
        'Recall@3': df_retrieval_summary['hit'].mean(),
        'Precision@3': df_retrieval_summary['precision@3'].mean(),
        'MRR': df_retrieval_summary['mrr'].mean(),
        'NDCG@3': df_retrieval_summary['ndcg@3'].mean(),
        'Wrong-Evidence Rate': df_retrieval_summary['wrong_evidence'].mean()
    }
])

reasoning_table = pd.DataFrame([
    {
        'Module': 'Reasoning Engine (Planner + PoT + Table Sandbox)',
        'Exact Match (EM)': df_execution_summary['em'].mean(),
        'F1 Score': df_execution_summary['f1'].mean(),
        'Numerical Exact-Match Accuracy': df_execution_summary['numerical_match'].mean(),
        'Arithmetic Error Rate': df_execution_summary['arithmetic_error'].mean(),
        'Tool Success Rate': df_execution_summary['tool_success'].mean()
    }
])

print("\n=== STEP 1 & 4 EVALUATION: ADVANCED RETRIEVAL PERFORMANCE ===")
print(benchmark_table.to_string(index=False))

print("\n=== STEP 2 & 3 EVALUATION: REASONING & PROGRAM-OF-THOUGHTS PERFORMANCE ===")
print(reasoning_table.to_string(index=False))